# 02 — Bronze | Incremental Ingestion with Auto Loader

**Project:** MLB Performance Intelligence Platform

Ingests every new JSON envelope landed in `/Volumes/mlb_intelligence/bronze/mlb_api_raw/` into
`bronze.mlb_api_responses`, one row per file.

| Item | Value |
|---|---|
| Reader | **Auto Loader** (`cloudFiles`, format `text`, `wholetext`), `trigger(availableNow=True)` |
| Checkpoint | `/Volumes/mlb_intelligence/metadata/checkpoints/bronze_mlb_api_responses/` — each file is ingested exactly once |
| Payload | Kept as the raw JSON string; parsing happens in Silver, so a schema change in the API never blocks ingestion |
| Lineage | Source file path, size and modification time, ingestion timestamp and batch id on every row |
| Write mode | Append-only (history of every extraction is preserved; Silver picks the latest per request) |

In [ ]:
import sys, importlib, os, uuid
_nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.insert(0, "/Workspace" + _nb.rsplit("/", 2)[0] + "/shared")
import config as C
import pipeline as P
for _m in (C, P):
    importlib.reload(_m)
from pyspark.sql import functions as F

RUN_ID = P.new_run_id()
CHECKPOINT = f"{C.CHECKPOINT_ROOT}/bronze_mlb_api_responses"
ENVELOPE = ("endpoint string, request_key string, request_url string, status_code int, "
            "extracted_at string, run_id string, payload_sha256 string, payload string")
ON_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ

In [ ]:
def to_bronze(raw):
    """Envelope text -> typed Bronze columns. A STRING field in from_json keeps the nested
    payload object as its raw JSON text."""
    env = F.from_json("value", ENVELOPE)
    return raw.select(
        env.endpoint.alias("endpoint"),
        env.request_key.alias("request_key"),
        env.request_url.alias("request_url"),
        env.status_code.alias("status_code"),
        F.to_timestamp(env.extracted_at).alias("extracted_at"),
        env.run_id.alias("extract_run_id"),
        env.payload_sha256.alias("payload_sha256"),
        env.payload.alias("payload"),
        F.col("_metadata.file_path").alias("_source_file"),
        F.col("_metadata.file_size").alias("_source_file_size"),
        F.col("_metadata.file_modification_time").alias("_source_modified_at"),
        F.current_timestamp().alias("_ingested_at"),
        F.lit(RUN_ID).alias("_batch_id"),
    )

with P.log_step(spark, RUN_ID, "bronze", "bronze_ingest", C.RAW_ROOT) as step:
    before = spark.table(C.BRONZE_API_RESPONSES).count() if spark.catalog.tableExists(C.BRONZE_API_RESPONSES) else 0
    if ON_DATABRICKS:
        stream = (spark.readStream.format("cloudFiles")
                  .option("cloudFiles.format", "text")
                  .option("wholetext", "true")
                  .option("pathGlobFilter", "*.json")
                  .option("cloudFiles.includeExistingFiles", "true")
                  .load(C.RAW_ROOT))
        (to_bronze(stream).writeStream
            .option("checkpointLocation", CHECKPOINT)
            .trigger(availableNow=True)
            .toTable(C.BRONZE_API_RESPONSES)
            .awaitTermination())
    else:
        # Local test harness only: batch read of files not yet ingested (same columns as above)
        batch = to_bronze(spark.read.option("wholetext", "true").option("pathGlobFilter", "*.json")
                          .option("recursiveFileLookup", "true").text(C.RAW_ROOT))
        if spark.catalog.tableExists(C.BRONZE_API_RESPONSES):
            batch = batch.join(spark.table(C.BRONZE_API_RESPONSES).select("_source_file"), "_source_file", "left_anti")
        batch.write.format("delta").mode("append").saveAsTable(C.BRONZE_API_RESPONSES)
    after = spark.table(C.BRONZE_API_RESPONSES).count()
    step.records = after - before
    step.message = f"{after - before} new file(s) ingested; table now has {after} rows"
    print(step.message)

spark.sql(f"""COMMENT ON TABLE {C.BRONZE_API_RESPONSES} IS
'Bronze: every MLB Stats API response landed in the raw archive, one row per file, payload kept as raw JSON. Append-only; Silver uses the latest extraction per (endpoint, request_key).'""")

In [ ]:
display(spark.sql(f"""
    SELECT endpoint, COUNT(*) AS files, COUNT(DISTINCT request_key) AS distinct_requests,
           ROUND(SUM(_source_file_size) / 1024 / 1024, 1) AS mb, MAX(extracted_at) AS last_extracted
    FROM {C.BRONZE_API_RESPONSES} GROUP BY endpoint ORDER BY endpoint"""))